# Looker

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Looker (Google Cloud Looker) is a BI and embedded-analytics platform whose semantic layer, LookML, is version-controlled code: it defines views, explores, dimensions, measures, and access rules, and Looker generates SQL against the warehouse at query time.

## Contents

<details>
<summary>39 sections - click to expand</summary>

- [What is Looker?](#what-is-looker)
- [What is LookML and why does it matter?](#what-is-lookml-and-why-does-it-matter)
- [How do you avoid fanout problems in Looker?](#how-do-you-avoid-fanout-problems-in-looker)
- [How does Looker handle data integration?](#how-does-looker-handle-data-integration)
- [Can you explain the concept of a LookML model in Looker?](#can-you-explain-the-concept-of-a-lookml-model-in-looker)
- [What is a dimension in Looker?](#what-is-a-dimension-in-looker)
- [How can you create a calculated field in Looker?](#how-can-you-create-a-calculated-field-in-looker)
- [What is a Looker Explore?](#what-is-a-looker-explore)
- [What is the purpose of a Looker dashboard?](#what-is-the-purpose-of-a-looker-dashboard)
- [Can Looker handle real-time data?](#can-looker-handle-real-time-data)
- [How does Looker handle data security?](#how-does-looker-handle-data-security)
- [What is the purpose of a Looker filter?](#what-is-the-purpose-of-a-looker-filter)
- [Can you explain the concept of Looker PDTs (Persistent Derived Tables)?](#can-you-explain-the-concept-of-looker-pdts-persistent-derived-tables)
- [How can you schedule data deliveries in Looker?](#how-can-you-schedule-data-deliveries-in-looker)
- [How does Looker handle data caching?](#how-does-looker-handle-data-caching)
- [Can Looker be used for data transformation and ETL processes?](#can-looker-be-used-for-data-transformation-and-etl-processes)
- [How does Looker handle data lineage and documentation?](#how-does-looker-handle-data-lineage-and-documentation)
- [What are some ways to optimize Looker performance?](#what-are-some-ways-to-optimize-looker-performance)
- [What is the purpose of Looker's Data Actions?](#what-is-the-purpose-of-lookers-data-actions)
- [What are some limitations of Looker?](#what-are-some-limitations-of-looker)
- [How does Looker handle data versioning and change management?](#how-does-looker-handle-data-versioning-and-change-management)
- [What is Looker's scheduling and alerting capability?](#what-is-lookers-scheduling-and-alerting-capability)
- [What is Looker's Explore API used for?](#what-is-lookers-explore-api-used-for)
- [How does Looker handle data governance and compliance?](#how-does-looker-handle-data-governance-and-compliance)
- [Can Looker handle multi-tenant data environments?](#can-looker-handle-multi-tenant-data-environments)
- [How does Looker handle data modeling for complex data structures?](#how-does-looker-handle-data-modeling-for-complex-data-structures)
- [How can Looker integrate with data orchestration tools like Apache Airflow?](#how-can-looker-integrate-with-data-orchestration-tools-like-apache-airflow)
- [What is the purpose of Looker's SQL Runner?](#what-is-the-purpose-of-lookers-sql-runner)
- [What is the difference between Looker's explore and join types?](#what-is-the-difference-between-lookers-explore-and-join-types)
- [What is the purpose of Looker's content validation feature?](#what-is-the-purpose-of-lookers-content-validation-feature)
- [Can you explain Looker's dynamic data masking feature?](#can-you-explain-lookers-dynamic-data-masking-feature)
- [What is the purpose of Looker's Liquid syntax in LookML?](#what-is-the-purpose-of-lookers-liquid-syntax-in-lookml)
- [What is the purpose of Looker's Development Mode?](#what-is-the-purpose-of-lookers-development-mode)
- [How do you design and rebuild PDTs for performance?](#how-do-you-design-and-rebuild-pdts-for-performance)
- [Things to remember: key numbers and limits](#things-to-remember-key-numbers-and-limits)
- [Things to remember: semantics to memorize](#things-to-remember-semantics-to-memorize)
- [Things to remember: mistakes that keep coming back](#things-to-remember-mistakes-that-keep-coming-back)
- [Real world use cases for Looker](#real-world-use-cases-for-looker)
- [Looker exercises](#looker-exercises)

</details>

Tip: the links open the sections on GitHub; in Jupyter or Colab, use the notebook's outline panel in the sidebar instead.

## What is Looker?

Looker is a business intelligence (BI) and data visualization platform that allows users to explore, analyze, and share data insights.

Architecturally, Looker is a SQL generator plus a semantic model, not a data store: queries run in the connected warehouse, and LookML decides which SQL runs. That is why governance, metric consistency, and performance tuning all live in the modeling layer rather than in individual dashboards.

Mental model: Looker is a SQL compiler, not a database. The LookML model is the program; a user's field picks, filters and pivots are the input; the output is one SQL statement per tile. Nothing is stored except the model metadata and the query result cache.

```text
field picks + filters + sorts
   -> Looker resolves every ${view.field} to its sql: expression
   -> builds SELECT, FROM/JOIN, WHERE, GROUP BY, HAVING, ORDER BY, LIMIT
   -> the dialect layer adapts functions (DATE_TRUNC, TIMESTAMP_SUB, casts)
   -> the warehouse runs it; Looker caches the rows and renders the tile
```

Worked example: an analyst picks `orders.created_month` and `orders.total_revenue`. Looker emits `SELECT DATE_TRUNC(created_at, MONTH) AS ..., SUM(amount) ... GROUP BY 1`. The same two clicks on BigQuery and on Snowflake produce different SQL text but the same numbers.

Common mistake: blaming Looker for a slow tile when the generated SQL is the problem. Always read the SQL tab first; the fix is usually a partition filter or one fewer join, not more Looker compute.

## What is LookML and why does it matter?

LookML is Looker's semantic modeling layer. It defines explores, views, joins, dimensions, measures, access rules, and reusable business logic. This helps teams centralize metric definitions instead of rebuilding them in each dashboard.

Example:

```text
measure: total_revenue {
  type: sum
  sql: ${amount} ;;
}
```

Strong LookML keeps BI consistent: "revenue" should mean the same thing across dashboards. A fuller example shows how dimensions, time frames, and measures compose in one view:

```text
view: orders {
  sql_table_name: analytics.fct_orders ;;
```


Trade-off: LookML makes metrics reusable and testable (a Git diff shows a metric change), but it adds a modeling step and a deployment workflow, so it slows down purely exploratory one-off analysis compared with a tool that lets analysts write raw SQL freely.

Three object types carry the model, and mixing them up is a frequent interview mistake.

```text
view:    one table or derived table -> dimensions, measures, filters, access grants
explore: a query surface -> one base view + joins + always_filter + cache policy
model:   connection + includes + explores + datagroups + week_start_day
```

A view is reusable; an explore is what a user actually opens. One view can appear in many explores, so a metric defined once behaves identically everywhere it is exposed.

How a generated query is built from a view: Looker walks the fields the user selected, pulls each `sql:` expression, qualifies it with the view alias, and wraps it in the dialect's aggregate.

```text
view: orders {
  sql_table_name: analytics.fct_orders ;;
  dimension: country { type: string sql: ${TABLE}.country ;; }
  measure: total_revenue { type: sum sql: ${TABLE}.amount ;; }
}
```

```sql
-- Generated from the view above
SELECT orders.country AS orders_country, SUM(orders.amount) AS orders_total_revenue
FROM analytics.fct_orders AS orders
GROUP BY 1;
```

Common mistake: putting business logic in a dashboard or a custom field instead of the view. It cannot be reused, reviewed, or tested, and it silently drifts from the governed metric.

## How do you avoid fanout problems in Looker?

Fanout happens when joins multiply rows and inflate measures. Prevent it by modeling correct join relationships, using aggregate-aware explores, pre-aggregating facts, or separating explores by grain.

Example:

```text
join: customers {
  type: left_outer
  relationship: many_to_one
  sql_on: ${orders.customer_id} = ${customers.customer_id} ;;
}
```

If the relationship is actually many-to-many, a bridge or aggregate table may be needed before exposing the data to users.

Two mechanisms protect you even when fanout is real. Symmetric aggregates make a `type: sum` correct across a join by aggregating the distinct primary keys first, and a `sql_distinct_key` makes a measure fanout-aware:

```text
measure: total_revenue {
  type: sum
  sql: ${TABLE}.amount ;;
  sql_distinct_key: ${order_id} ;;   # deduplicate at order grain before summing
}
```


Fanout with `type: count_distinct` is a common trap: a count of distinct rows survives the join, but a `type: sum` doubles unless a distinct key is given. When NOT to rely on symmetric aggregates: the measure is over a pre-aggregated table where the primary key is no longer at the row grain; there, aggregate before joining or use separate explores.

Symmetric aggregates are the mechanical fix. When a measure sits on a view with a `primary_key` and the join is declared `one_to_many`, Looker does not sum across the joined rows directly. It computes the measure per distinct primary key, then sums those values in an outer query, so each order is counted once no matter how many item rows it produced.

```sql
-- The shape Looker emits for a fanout-safe sum
SELECT orders.country AS orders_country,
       SUM(order_items_sum) AS orders_total_revenue
FROM (
  SELECT orders.order_id, orders.country, SUM(orders.amount) AS order_items_sum
  FROM analytics.fct_orders AS orders
  GROUP BY 1, 2
) AS symmetric_aggregate
GROUP BY 1;
```

Worked example: two orders with amounts 400 and 600, each with three item rows. A naive `SUM(orders.amount)` over the join returns 3000 because every order row repeats three times. With `primary_key: yes` on `orders.order_id` and `relationship: one_to_many`, Looker returns 1000.

Common mistakes: forgetting `relationship:` on the join (Looker cannot apply symmetric aggregates), or writing `type: sum` over a derived table that has no declared primary key. In both cases the fanout survives and every ratio built on that measure is wrong.

## How does Looker handle data integration?

Looker connects to various data sources using database dialects and APIs, allowing it to integrate with different data platforms such as SQL databases, cloud storage, and data warehouses.

Looker talks SQL to a warehouse; it does not ingest data. Integration therefore means: a connection (with a service account or JDBC/ODBC credentials), a dialect (BigQuery, Snowflake, Redshift, Postgres, Databricks, etc.), and a LookML project whose `sql_table_name` points at warehouse objects. For files in a data lake, you expose them through the warehouse (external tables, Iceberg, BigLake, Delta sharing).

```text
connection: "acme_bigquery"
```


The connection is a first-class object, and its settings appear in LookML behaviour. The important fields are the dialect, the PDT dataset (where derived tables are physically written), the temp dataset (scratch space for large queries), the maximum connection pool (concurrency, not correctness) and the service account the warehouse sees.

```text
connection: "acme_bigquery"
# dialect drives function names and cast syntax in generated SQL
# PDT dataset must be writable and kept clean; stale PDTs cost storage
```

Gotcha: every connection is its own cache namespace and its own permissions boundary. A query that works on one connection can fail on another purely because of dialect differences, and a user attribute used in an access filter is evaluated per connection.

Common mistake: granting Looker a service account with write access to production schemas. The connection only needs read on the marts plus write on the PDT and temp datasets.

## Can you explain the concept of a LookML model in Looker?

LookML is Looker's modeling language that defines the relationship between data tables and creates reusable dimensions and measures. A LookML model is a collection of files that describe the structure and logic of your data.

A project has `.model.lkml` files (which define explores and the connection), `.view.lkml` files (which define dimensions and measures over one table or derived table), plus manifests, refinements, and data tests.

```text
# model file: acme.model.lkml
connection: "acme_bigquery"
```


A project is a Git repository of files, and `include` decides what the model can see. Nothing is global by accident.

```text
acme/
  acme.model.lkml          connection, includes, explores, datagroups
  views/orders.view.lkml
  explores/orders.explore.lkml
  manifest.lkml
  tests/orders_unique_test.lkml
```

```text
include: "/views/*.view.lkml"
include: "/explores/*.explore.lkml"
```

Model-level settings apply to every explore in the file: `connection`, `week_start_day`, `persist_with` defaults, `datagroup` definitions, and `test` declarations. A datagroup defined in one model file is not visible in another unless it is included.

Common mistake: a view exists in the project but is not included by any model, so it never appears in the UI and the developer assumes a caching bug.

## What is a dimension in Looker?

A dimension in Looker represents a field or attribute in your data. It provides context and adds descriptive information to your analysis.

Dimension types include `string`, `number`, `yesno`, `tier`, `zipcode`, `distance`, and the common `time`. A time dimension is usually declared with `dimension_group` so Looker generates `orders.created_date`, `orders.created_month`, and so on automatically.

```text
dimension: segment {
  type: string
  sql: ${TABLE}.segment ;;
  suggest_explore: customers     # autocomplete values from a small explore
}
```


Dimensions describe rows; measures aggregate them. The three declarations that matter most are the primary key, the time dimension group, and the SQL expression.

```text
dimension: order_id { primary_key: yes type: string sql: ${TABLE}.order_id ;; }
dimension: status   { type: string sql: ${TABLE}.status ;; }
dimension_group: created {
  type: time
  timeframes: [date, week, month, quarter, year]
  datatype: timestamp
  sql: ${TABLE}.created_at ;;
}
```

`dimension_group` generates one field per time frame, so `orders.created_month` exists without extra code.

Worked example: `dimension: status { type: string sql: ${TABLE}.status }` produces `orders.status` in the field picker and `orders.status AS orders_status` in SQL. Changing `type` to `number` changes the cast, the filter operator set, and the sort order.

Common mistakes: writing `sql: status` instead of `${TABLE}.status` (the unqualified name breaks inside joins), and omitting `primary_key: yes` on the grain column, which disables symmetric aggregates for that view.

## How can you create a calculated field in Looker?

In Looker, calculated fields can be created using LookML or Looker's Explore interface. LookML allows for more advanced calculations and reusable logic.

Looker distinguishes three layers of calculation, and interviews probe the difference:

+ LookML measure: runs in the warehouse SQL, reusable everywhere (`type: number`, `sql: ...`).
+ Table calculation: runs in Looker on the result set (percent of total, running total, offset).
+ Custom field in Explore: ad-hoc, user-scoped, not governed.

```text
measure: gross_margin_pct {
  type: number
  value_format_name: percent_2
  sql: (${total_revenue} - ${total_cost}) / NULLIF(${total_revenue}, 0) ;;
}
```

```text
# Table calculation (Looker expression, not SQL): running revenue within the view
running_total(${orders.total_revenue})
```


Trade-off: a measure is governed and consistent but requires a Git deploy; a table calculation is instant but exists only in that tile and can silently disagree with the governed metric.

The measure `type` is the SQL decision, not a formatting choice. `sum`, `average`, `count`, `count_distinct`, `min`, `max` and `median` become warehouse aggregates; `number` becomes a scalar expression over other measures.

```text
measure: order_count      { type: count sql: ${TABLE}.order_id ;; }
measure: customer_count   { type: count_distinct sql: ${TABLE}.customer_id ;; }
measure: total_revenue    { type: sum sql: ${TABLE}.amount ;; }
measure: gross_margin_pct {
  type: number
  visible_if_any: [total_revenue]
  sql: (${total_revenue} - ${total_cost}) / NULLIF(${total_revenue}, 0) ;;
}
```

Worked example: a ratio must be built from summed components, never from an average of per-row percentages. `SUM(margin) / SUM(revenue)` is correct; `AVG(margin_pct)` is not additive across days or countries.

Common mistake: using `type: average` for a metric that will later be rolled up or filtered, then discovering that averaging averages is mathematically wrong. Model the numerator and denominator and divide.

## What is a Looker Explore?

A Looker Explore is a tool within Looker that allows users to interactively explore and analyze data using a graphical user interface.

An Explore is the query surface: users pick fields from one or more joined views, apply filters, and Looker writes the SQL. What defines a good Explore is not the UI but the model behind it: correct `relationship`, `fields` curation, `always_filter`/`conditionally_filter` for cost control, and `access_filter` or `required_access_grants` for security.

```text
explore: orders {
  always_filter: { filters: [orders.created_date: "last 30 days"] }
  conditionally_filter: {
    filters: [orders.country: "US"]
    unless: [orders.created_date, orders.region]
  }
}
```

The query lifecycle is the same for the UI, the API and a scheduled delivery.

```text
1. the user picks fields, filters, sorts and a row limit
2. Looker checks access: access_filter, required_access_grants, model sets
3. Looker resolves fields to sql: expressions and builds one statement
4. symmetric aggregates and aggregate awareness may rewrite the FROM clause
5. the statement runs on the connection; the result is cached under a cache key
6. the tile renders; the next identical request is served from cache
```

Step 4 is why two explores over the same table can return different SQL for the same measure.

An explore is a product decision as much as a modeling one: `fields:` curates the picker, `hidden: yes` removes internals, `always_filter` guards the partition, and `label`/`description` teach the user.

Common mistake: joining eight views into one explore "so anything is possible". Users then build accidental fanout and cartesian products, and every tile scans everything. Prefer several narrow explores over one universal one.

## What is the purpose of a Looker dashboard?

A Looker dashboard is a customizable collection of visualizations and reports that provide a consolidated view of key metrics and data insights.

Dashboards are layout + tiles + filters; each tile is a stored query. Dashboard-level filters (`listen` blocks) push values into tiles, and `dashboard:element` filters can drive cross-tile behavior.

```text
# Dashboard defined as LookML (dashboards can be code, not just UI artifacts)
- dashboard: revenue_overview
  title: Revenue Overview
  layout: newspaper
  filters:
    - name: date
      title: Date Range
      type: date_filter
      default_value: "90 days"
  elements:
    - name: revenue_by_country
      title: Revenue by Country
      model: acme
      explore: orders
      type: looker_bar
      fields: [orders.country, orders.total_revenue]
      listen: { date: orders.created_date }
```

A dashboard owns no data: each tile is a saved query with its own model, explore, fields and filters. Dashboard filters broadcast to tiles through `listen` mappings, and a cross-filter narrows other tiles from a click.

```text
- dashboard: revenue_overview
  filters:
    - name: date
      type: date_filter
      default_value: "90 days"
  elements:
    - name: revenue_by_country
      model: acme
      explore: orders
      fields: [orders.country, orders.total_revenue]
      listen: { date: orders.created_date }
```

Gotcha: dashboard load time is the sum of its tiles divided by the cache hit rate. Twelve uncached tiles and fifty users is up to six hundred warehouse queries per refresh. A date filter with no `default_value` makes every tile scan all history.

Common mistake: building the dashboard first and the model later. The dashboard is the cheap part; the model determines whether it is fast and consistent.

## Can Looker handle real-time data?

Looker primarily works with batch data processing and is not designed for real-time data analysis. However, you can set up scheduled data refreshes to get near real-time insights.

The accurate answer: Looker can query any warehouse table at any freshness, so "real time" depends entirely on the warehouse. What Looker controls is the cache. Turning caching off (`max_cache_age: 0` or a short `datagroup`) gives live results at the cost of warehouse load.

```text
# Per-model cache policy
datagroup: orders_default {
  max_cache_age: "1 hour"
}
```


Trade-off: disabling cache on a dashboard with 12 tiles and 50 concurrent users multiplies warehouse load 12x per user interaction. The right pattern is a fast-changing narrow explore for operational dashboards and a cached, modeled explore for analysis.

Freshness is a chain, and Looker controls only one link: warehouse ingest latency, then cache age, then the user's refresh.

```text
source -> ingestion (seconds to hours)  ->  warehouse table
       -> Looker cache (datagroup or max_cache_age)
       -> the tile the user is looking at
```

If ingest lags ten minutes, a zero-second cache buys nothing.

Worked example: an operations dashboard sets `datagroup: ops_5m { max_cache_age: "5 minutes" }` and an `always_filter` on today's partition. Eight tiles and ten users then cost at most sixteen queries per minute, and each query touches one partition.

Common mistake: disabling cache entirely to chase real time, then wondering why the warehouse queue grows. Short cache plus a narrow partition filter is almost always the right operational answer.

## How does Looker handle data security?

Looker offers various security features, such as user access controls, data encryption, and integration with identity providers for authentication and authorization.

Layers of security:

+ Authentication: SAML/OAuth/OIDC/Google SSO, LDAP, or email/password (discouraged).
+ Authorization: roles -> permission sets + model sets; `access_grant` and `required_access_grants` at field level.
+ Row-level security: `access_filter` or user attributes injected into SQL with Liquid.
+ Data: all queries run in the warehouse with the connection's service account; Looker stores no customer data beyond query cache.

```text
# Row-level security via a user attribute: each user sees only their region
explore: orders {
  access_filter: {
    field: orders.region
    user_attribute: region
  }
}
```


An `access_filter` injects a predicate built from a user attribute at query time. Looker reads the attribute value from the user (or the SSO embed signature), quotes it, and adds `WHERE field = value` to every query in that explore.

```text
explore: orders {
  access_filter: {
    field: orders.region
    user_attribute: region
  }
}
```

Gotcha: the filter fails closed. A user with no `region` attribute gets no rows, not all rows, which is the safe default but surprises administrators. A user with several allowed regions needs an attribute of type "string" holding a list, or a separate access filter per region.

Common mistake: testing row-level security as an administrator, who usually has a bypass attribute. Always test as a real user in the target group.

## What is the purpose of a Looker filter?

A Looker filter allows users to dynamically limit the data displayed in a report or visualization based on specific criteria.

Filters are translated into SQL `WHERE` clauses (or `HAVING` for measures). Looker adds filter suggestions, `always_filter` defaults, and `conditionally_filter` guards. A subtle point: filtering on a measure uses `HAVING`, which cannot use an index or partition pruning, so prefer dimension filters on partitioned columns.

```sql
-- What a typical Looker query compiles to (BigQuery dialect)
SELECT
  orders.country AS orders_country,
  SUM(orders.amount) AS orders_total_revenue
FROM `acme-data-prod.curated.fct_orders` AS orders
WHERE (orders.created_at >= TIMESTAMP('2026-07-06')
       AND orders.created_at < TIMESTAMP('2026-10-04'))
  AND orders.country IN ('DE', 'FR')
GROUP BY 1
ORDER BY 2 DESC
LIMIT 500;
```

Filter syntax maps to SQL clauses, and the mapping decides performance. Dimension filters become `WHERE`; measure filters become `HAVING`; a filter on a time frame becomes a range predicate on the raw time column.

```text
filters: [orders.status: "complete,shipped"]
-> WHERE orders.status IN ('complete', 'shipped')

filters: [orders.total_revenue: ">1000"]
-> HAVING SUM(orders.amount) > 1000   -- after aggregation, no pruning
```

Looker offers three defaulting mechanisms: `filters:` (optional UI filter), `always_filter` (must be set, prefilled), and `conditionally_filter` (required unless another filter is present). The last one is the standard cost guard on a large fact table.

Common mistake: letting users filter a measure on a high-volume explore. The `HAVING` clause runs after the full scan, so the query is as expensive as the unfiltered one. Push the condition into a dimension or a PDT instead.

## Can you explain the concept of Looker PDTs (Persistent Derived Tables)?

Looker PDTs are precomputed tables created in the database to improve query performance. They are especially useful for complex or resource-intensive calculations.

A PDT is a `derived_table` that Looker materializes with `CREATE TABLE AS SELECT` (or a materialized view), then rebuilds on `persist_for` or `datagroup` triggers. PDTs can be native (in the warehouse's scratch schema) or written to a schema your team owns.

```text
view: customer_lifetime_metrics {
  derived_table: {
    sql:
      SELECT
        o.customer_id,
        MIN(o.created_at) AS first_order_at,
        MAX(o.created_at) AS last_order_at,
        COUNT(DISTINCT o.order_id) AS orders,
        SUM(o.amount) AS lifetime_revenue
      FROM analytics.fct_orders o
      GROUP BY 1 ;;
    persist_for: "24 hours"
    datagroup_trigger: orders_default
    cluster_keys: ["customer_id"]
  }
```


Trade-off: a PDT trades warehouse storage and build time for query speed and gives you a governed place for expensive logic, but each PDT is another table to monitor, and a `persist_for` that is too short causes rebuild storms while one that is too long silently serves stale data.

There are four shapes, and the interview answer is the difference between them.

```text
ephemeral derived_table          no persistence rule -> inlined as a subquery/CTE
view-based derived_table         a view over a subquery, still inlined unless persisted
persistent derived table (PDT)   has persist_for / datagroup_trigger / sql_trigger
incremental PDT                  persistent + incremental: yes, appends new rows
```

A `derived_table` only becomes a physical table when a persistence rule is present. Without one it is folded into every query that uses it, which is fine for small logic and disastrous for a 50M-row aggregation.

Looker writes a PDT by building into a temporary name and then swapping it into place against a designer/reader split, so a reader never sees a half-built table. The connection's PDT dataset must exist and be writable, and a failed build leaves the previous version in place.

```text
view: customer_lifetime_metrics {
  derived_table: {
    sql: SELECT customer_id, SUM(amount) AS lifetime_revenue
         FROM analytics.fct_orders GROUP BY 1 ;;
    datagroup_trigger: orders_default
    cluster_keys: ["customer_id"]
  }
}
```

Common mistake: giving every PDT the same datagroup trigger. When the ETL signal changes, all of them rebuild at once, the warehouse queue saturates, and the dashboards that depend on them are slow exactly when users arrive in the morning. Stagger triggers by freshness tier.

## How can you schedule data deliveries in Looker?

Looker provides scheduling options that allow you to automate the delivery of reports and dashboards via email, Slack, or other destinations.

Schedules can be created in the UI or via the API, with formats (PDF, PNG, CSV, XLSX), destinations (email, SFTP, S3/GCS, webhook, Slack), and run-as owner. API-driven schedules are how you keep delivery configuration in version control.

In [ ]:
# ILLUSTRATIVE - requires looker_sdk: install to run, or read on.
# Create a scheduled plan with the Looker API 4.0
import looker_sdk

sdk = looker_sdk.init40()   # reads LOOKERSDK_* env vars

plan = sdk.create_scheduled_plan(
    body={
        "name": "Weekly revenue CSV to finance",
        "look_id": 1234,
        "crontab": "0 7 * * 1",
        "enabled": True,
        "run_as_recipient": False,
        "scheduled_plan_destination": [{
            "format": "csv",
            "type": "email",
            "address": "finance-ops@acme.com",
            "apply_formatting": False,
        }],
    }
)
print(plan.id)

A schedule is a saved query plus a destination, a format, a frequency and a run-as identity. Advanced schedules add cron expressions, time zones, format variants and multiple destinations.

```text
delivery options
  format: pdf, png, csv, xlsx, json, txt
  destination: email, sftp, s3/gcs, webhook, slack, teams
  run_as: owner (one result for everyone) or recipient (row-level security per person)
```

Gotcha: the run-as choice is a security decision, not a convenience. Run-as-owner delivers the owner's filtered rows to every recipient; run-as-recipient applies each person's user attributes, which is correct for regional or tenant reports but multiplies the queries.

Common mistake: a scheduled PDF with no date filter, which mails the full history every morning and grows until the delivery times out.

## How does Looker handle data caching?

Looker intelligently caches query results to improve performance. The caching strategy can be customized based on the frequency and volatility of the underlying data.

Caching is per-query-result and controlled by `datagroup` policies (preferred), `persist_with`, or per-explore/connection overrides. `datagroup` is preferable because it ties cache invalidation to the warehouse load: when the ETL sets the datagroup's trigger table, Looker invalidates only the affected explores.

```text
# Datagroup whose SQL trigger is the date the ETL updated the table
datagroup: orders_default {
  sql_trigger: SELECT MAX(etl_run_at) FROM analytics.etl_audit WHERE table_name = 'fct_orders' ;;
  max_cache_age: "24 hours"
  description: "Invalidates when the orders load completes"
}
```

Trade-off: long cache reduces warehouse cost and speeds dashboards, but users who need to see a just-loaded row must either use a realtime explore or looker can be told to bypass cache; `cache` is also not shared across users with different row-level-security filters, so warehouse load scales with distinct security contexts.

The cache key is the full query plus the security context. Two users with different `access_filter` values never share a result, which is why row-level security and cache efficiency pull against each other.

```text
cache key = generated SQL + filters + sorts + user attributes used in the model
datagroup: orders_default {
  sql_trigger: SELECT MAX(etl_run_at) FROM analytics.etl_audit WHERE table_name = 'fct_orders' ;;
  max_cache_age: "24 hours"
}
explore: orders { persist_with: orders_default }
```

`persist_with` binds an explore to a datagroup; `max_cache_age` is the fallback ceiling when no trigger fires.

Worked example: a dashboard is run 400 times a day by 20 users, all with the same region attribute. With an hourly datagroup and a 24-hour cache age, the warehouse sees at most 24 queries instead of 400.

Common mistake: setting `max_cache_age` without a datagroup. The dashboard serves results loaded just before the ETL ran for the whole window, and nobody can explain the stale number. Tie invalidation to the load, not to a timer.

## Can Looker be used for data transformation and ETL processes?

Looker is primarily focused on data visualization and exploration. While it provides some transformation capabilities, it is not a dedicated ETL tool.

Looker's transformation surface is PDTs and SQL Runner, plus Looker's integration with dbt (the recommended place for warehouse transformations). Use dbt/Dataflow/Spark for the heavy pipeline, and Looker for the last-mile aggregation that is too expensive to run per query.

```text
Transform where it belongs
  ingestion + cleaning + conformed models  -> dbt / Spark / Dataflow
  last-mile aggregates and wide tables     -> Looker PDT, materialized view
  ad-hoc exploration                       -> Explore with table calculations
```

The split that works in production: dbt (or Spark/Dataflow) owns ingestion, cleaning and conformed dimensions; Looker PDTs own the last-mile aggregation that is too expensive to recompute per query. Looker's datagroups trigger rebuilds but do not schedule arbitrary pipelines.

```text
dbt/Spark:   raw -> staging -> marts (tested, versioned, documented upstream)
Looker PDT:  marts -> expensive rollup / wide table reused by many tiles
Explore:     ad-hoc cuts and table calculations on top
```

Gotcha: chaining PDTs into each other creates a dependency graph inside Looker that has no upstream lineage and rebuilds in unpredictable order. If a transform must run before another, it belongs in the warehouse scheduler.

Common mistake: moving a slow join into a PDT to make one dashboard fast, then discovering three other explores also needed it and now maintain two definitions.

## How does Looker handle data lineage and documentation?

Looker allows you to document the logic and definitions of your data models using LookML. It also provides options to track data lineage and maintain documentation within the platform.

LookML is self-documenting: `description`, `label`, `group_label`, `hidden`, and `tags` are surfaced in the UI; `/projects` shows LookML dependencies between views and explores; and the Admin panel exposes the SQL of any query. For end-to-end lineage across the warehouse, pair Looker with dbt docs or a catalog (DataHub, OpenMetadata, Dataplex) via the API.

```text
measure: total_revenue {
  label: "Total Revenue (net of returns)"
  description: "Sum of order amounts after refunds. Source: analytics.fct_orders.amount"
  tags: ["finance", "certified"]
  type: sum
  sql: ${TABLE}.amount ;;
}
```

Documentation lives next to the SQL. `label`, `description`, `group_label`, `tags` and `hidden` are rendered in the field picker, and `/projects` exposes the dependency graph between views and explores.

```text
measure: total_revenue {
  label: "Total Revenue (net of returns)"
  description: "SUM(fct_orders.amount) after refunds. Owner: finance-analytics."
  tags: ["finance", "certified"]
  type: sum
  sql: ${TABLE}.amount ;;
}
```

Gotcha: a description is prose, so it drifts when the SQL changes. Certify the few metrics that matter, review description changes in the same pull request as the SQL, and use the content validator to confirm nothing downstream broke.

Common mistake: documenting only in a wiki. The wiki is invisible from the field picker, so users still ask which revenue definition is correct.

## What are some ways to optimize Looker performance?

Performance optimization in Looker can be achieved through techniques such as optimizing SQL queries, utilizing caching, using PDTs strategically, and organizing your LookML models efficiently.

```text
Optimization order (cheapest first)
1. Fix the warehouse: partition/cluster the tables Looker scans, add the right sort keys
2. Reduce bytes: remove unneeded joins, avoid SELECT *, use aggregate awareness
3. Cache: datagroup tied to the ETL, longer max_cache_age for stable explores
4. PDT / materialized view for expensive repeated aggregations
5. Limit result sets: row limits, always_filter on a date partition
6. Only then add warehouse compute
```

```text
# Aggregate awareness: transparently use a pre-aggregated rollup when possible
explore: orders {
  aggregate_table: daily_rollup {
    query: {
      dimensions: [orders.created_date, orders.country]
      measures: [orders.total_revenue, orders.order_count]
    }
    materialization: {
      sql_trigger_value: SELECT MAX(etl_run_at) FROM analytics.etl_audit WHERE table_name='fct_orders' ;;
    }
  }
}
```

The first artifact to inspect is the generated SQL, not the Looker settings. Open the Explore, click the SQL tab, and read it as if a colleague wrote it.

```text
what to look for
  SELECT * or fields you did not ask for        -> model exposes too much
  no predicate on the partition column          -> full scan
  a function wrapping the partition column      -> pruning disabled
  a many_to_many join                           -> fanout and row explosion
  ORDER BY with no LIMIT on a huge result       -> warehouse sort
```

Aggregate awareness is the model-level lever: a rollup that Looker can substitute when the requested dimensions, measures and filters all exist in it. It is transparent to the user and requires no dashboard change.

```text
explore: orders {
  aggregate_table: daily_rollup {
    query: {
      dimensions: [orders.created_date, orders.country]
      measures: [orders.total_revenue, orders.order_count]
    }
    materialization: { datagroup_trigger: orders_default }
  }
}
```

Common mistake: adding a small `LIMIT` to make a query feel fast. The warehouse still scanned and aggregated the full table; only the bytes returned to the browser shrank. Always ask what the engine read, not what the browser received.

## What is the purpose of Looker's Data Actions?

Looker's Data Actions enable users to take actions directly from their data, such as updating records, creating tasks, or triggering external processes.

Actions are configured in the Action Hub (Slack, email, Google Sheets, Segment, Zapier, custom webhooks) and can be invoked from a dashboard tile or a data table row. They are a write path from the semantic layer, so treat them as production integrations: authenticate, validate, and audit.

```text
# Invoke a custom webhook action from a LookML-tagged dashboard tile
action: {
  name: "create_jira_ticket"
  label: "Escalate to Jira"
  url: "https://hooks.acme.internal/looker/jira"
  form: {
    ticket_title: { label: "Title" }
    priority: { label: "Priority", type: "select", options: ["P1", "P2", "P3"] }
  }
}
```

An action is declared in LookML with a URL and a form, then attached to a dashboard tile or a table row. Looker posts the selected row values and the form fields to the endpoint.

```text
action: {
  name: "create_jira_ticket"
  label: "Escalate to Jira"
  url: "https://hooks.acme.internal/looker/jira"
  form: {
    ticket_title: { label: "Title" }
    priority: { label: "Priority", type: "select", options: ["P1", "P2", "P3"] }
  }
}
```

Gotcha: actions write to external systems, so they need scoped credentials, idempotency keys and an audit log. A double-click must not create two tickets.

Common mistake: putting an API token in the action's LookML so it lands in Git. Secrets belong in the Action Hub configuration, which is admin-managed and not versioned with the model.

## What are some limitations of Looker?

Some limitations of Looker include its focus on batch processing, limited real-time capabilities, and the need for underlying data infrastructure such as data warehouses.

The honest list: (1) it is expensive per user and pricing is negotiated, not list; (2) it requires a performant SQL warehouse, so cost is Looker + warehouse; (3) no native streaming/real-time push (it polls the warehouse); (4) LookML has a learning curve and the developer workflow (IDE, Git, deploy) is not beginner-friendly; (5) visual customization is weaker than Tableau/Power BI for pixel-perfect layouts; (6) row-level security adds warehouse load because cache is per security context. When NOT to use it: a single-team ad-hoc analysis shop with no warehouse, or a scenario needing sub-second push updates.

The honest scope: Looker is an access, semantic and governance layer, not storage, not a scheduler, not a rendering engine. Each of those comes from somewhere else.

```text
storage + compute   -> the warehouse
pipelines           -> dbt / Airflow / Spark / Dataflow
pixel-perfect layout-> weaker than Tableau or Power BI
streaming push      -> not supported; Looker polls the warehouse
```

The cost model is the other limit: per-user licensing plus warehouse spend, and the two move together. Row-level security multiplies cache entries, which multiplies warehouse spend without changing the license count.

Common mistake: adopting Looker to fix a data quality problem. It will faithfully and visibly surface bad data faster than before.

## How does Looker handle data versioning and change management?

Looker allows you to version control your LookML models and manage changes using Git or other version control systems.

Each project is a Git repository; Development Mode creates a branch, the IDE validates and diffs changes, and deploying to Production is a commit/merge. Content (dashboards, Looks, schedules) is *not* in Git by default — export it with the API or use Looker's content validation and the `/content` endpoints if you need content-as-code.

```bash
# List projects and their Git state with the API
curl -s -H "Authorization: token $LOOKER_TOKEN" \
  "https://acme.looker.com/api/4.0/projects" \
  | python -c "import sys,json; [print(p['id'], p['git_remote_url'], p.get('uses_git')) for p in json.load(sys.stdin)]"
```

Each project is a Git repository; the Looker IDE is a client for it. Development Mode holds a personal branch and keeps production untouched until Deploy. Rollback is a Git revert plus a deploy.

```text
dev branch (personal) -> shared feature branch -> main -> Deploy to Production
production LookML is read-only in the IDE by design
```

Gotcha: the deployed model is not stamped per-query. The System Activity model records which model ran, but reproducing last month's number requires the Git tag or commit that was deployed then. Tag releases if the business needs auditability.

Common mistake: editing a shared feature branch directly instead of one's own dev branch, which makes two people's uncommitted work overlap.

## What is Looker's scheduling and alerting capability?

Looker's scheduling and alerting feature allows you to schedule reports, dashboards, and data deliveries at specific intervals. You can also set up alerts based on predefined thresholds or conditions.

Looker alerts run a saved query on a schedule and notify when a condition holds (for example, `revenue < threshold`), sending to email or a webhook. Alerts are per-user in the UI (and API-manageable), and a common trap is alert fatigue: alert on the metric a human can act on, with a link to the dashboard and a runbook.

```text
# Alert condition example: page when daily revenue drops below the floor
# condition: orders.total_revenue < 250000
# comparison: less than
# schedule: every 6 hours during business hours
# destination: webhook -> PagerDuty
```

An alert is a saved query, a comparison condition, a schedule and a destination. Looker evaluates the condition on each run and notifies only when it holds (or on every run, depending on the setting).

```text
condition: orders.total_revenue < 250000
comparison: less than
schedule: every 6 hours during business hours
destination: webhook -> PagerDuty
```

Gotcha: the alert queries the whole explore unless it carries its own filter, so an alert on an unfiltered fact table is a scheduled full scan. Give alerts the same partition guard as a dashboard.

Common mistake: alerting on a raw metric with no owner or runbook. The alert fires, nobody knows what to do, and within a week it is muted.

## What is Looker's Explore API used for?

Looker's Explore API allows developers to programmatically query and retrieve data from Looker Explore instances, enabling integration with external applications.

The Explore API (`run_inline_query`) returns governed query results as JSON/CSV/HTML and is used to power embedded analytics, reverse ETL, and data-app backends. It is preferable to querying the warehouse directly because filters, joins, and row-level security are applied by the model.

```bash
# Inline query via REST: same semantics as the Explore UI, no SQL duplication
curl -s -X POST -H "Authorization: token $LOOKER_TOKEN" -H "Content-Type: application/json" \
  -d '{"model":"acme","view":"orders","fields":["orders.country","orders.total_revenue"],
       "filters":{"orders.created_date":"30 days"},"limit":"50"}' \
  "https://acme.looker.com/api/4.0/queries/run/json" | head -c 300
```

`run_inline_query` executes the same query lifecycle as the UI: access filters apply, the model builds the SQL, and the result comes back as JSON, CSV or HTML. `run_look` does the same for a saved Look.

```text
POST /api/4.0/queries/run/json
{"model":"acme","view":"orders",
 "fields":["orders.country","orders.total_revenue"],
 "filters":{"orders.created_date":"30 days"},
 "limit":"50"}
```

Gotcha: the inline query is subject to the API user's permissions, not the end user's, unless you use `run_as_user` (which requires the `sudo` permission). That is the usual source of "the API shows more than the dashboard".

Common mistake: calling the API with the admin service account from a product backend, which bypasses tenant row-level security entirely. Use embed SSO or run-as-user instead.

## How does Looker handle data governance and compliance?

Looker offers features for data access controls, auditing, and compliance with regulations like GDPR and HIPAA. It allows administrators to manage user permissions and track data usage.

Governance controls: roles/permission sets, model sets, field-level access grants, user attributes for row-level security, content validation for broken/errored content, system activity explores and audit logs (`System Activity` model), and data retention controls for query cache and history. Compliance certifications come from the platform (SOC 2, ISO 27001, HIPAA-eligible under a BAA) and from the warehouse, so the data residency story is the warehouse's.

```text
access_grant: see_pii {
  user_attribute: pii_access
  allowed_values: ["yes"]
}
```


Governance is expressed in the model and enforced by the warehouse. An `access_grant` is a named condition on a user attribute; `required_access_grants` attaches it to fields and explores.

```text
access_grant: see_pii {
  user_attribute: pii_access
  allowed_values: ["yes"]
}
dimension: email {
  sql: ${TABLE}.email ;;
  required_access_grants: [see_pii]
}
```

Gotcha: an access grant hides a field in the UI, it does not remove the column from the underlying table. A user with SQL Runner access can still select it. Column-level protection must also exist in the warehouse (masking policies, policy tags).

Common mistake: assuming that hiding a field in LookML is data masking. It is UI and query-build protection, nothing more.

## Can Looker handle multi-tenant data environments?

Looker supports multi-tenant data environments, allowing different users or groups to access and analyze their specific datasets securely within the same instance.

Multi-tenancy is done with user attributes + `access_filter` (or hybrid: a separate warehouse per large tenant and a shared one for the long tail). The key design question is cache efficiency: because filters differ per tenant, cached results are per security context, so 500 tenants means 500 cache entries for the same dashboard unless you share a tenant-agnostic rollup.

```text
explore: usage {
  access_filter: {
    field: usage.tenant_id
    user_attribute: tenant_id
  }
}
```

Two proven architectures. Shared-everything: one warehouse, one model, `access_filter` on `tenant_id`. Hybrid: large tenants get their own schema or warehouse, the long tail shares one.

```text
shared:  one dashboard, N security contexts, N cache entries
hybrid:  premium tenants isolated by connection, tail cached once
```

Gotcha: cache efficiency. Because the cache key includes the security context, 500 tenants running the same dashboard produce up to 500 warehouse queries per refresh. If the tenant filter can be pushed to a shared tenant-agnostic rollup, do that.

Common mistake: one model file per tenant. It multiplies the modeling surface, the test surface and the deploy risk; parameterize instead.

## How does Looker handle data modeling for complex data structures?

Looker's LookML modeling language provides flexibility to handle complex data structures by defining relationships, joins, and transformations using LookML files.

Complex structures are handled by flattening in the warehouse (arrays, JSON), then modeling with `derived_table`, `extends`, and `refinements`. `extends` lets you build a base view (all dimensions) and specialized child views (e.g., a `orders_secure` view that hides PII), while `refinement` lets a downstream project add access grants without forking the base project.

```text
view: orders_base {
  dimension: order_id { primary_key: yes type: string sql: ${TABLE}.order_id ;; }
  measure: total_revenue { type: sum sql: ${TABLE}.amount ;; }
}
```


`extends` builds a base view and specialized children; a refinement adds behaviour to a view defined in another project without forking it.

```text
view: orders_base {
  dimension: order_id { primary_key: yes type: string sql: ${TABLE}.order_id ;; }
  measure: total_revenue { type: sum sql: ${TABLE}.amount ;; }
}
view: orders_secure {
  extends: [orders_base]
  dimension: customer_email { hidden: yes }
}
```

```text
# downstream project refinement
view: +orders_base {
  required_access_grants: [orders_access]
}
```

Gotcha: `extends` copies fields at parse time, so a later change to the base view does not propagate to children that were generated from an older deploy. Refinements (`+view`) do follow the base but can only add or override in defined ways.

Common mistake: extending a view to change one measure, then maintaining two divergent definitions of revenue. Use a refinement or a parameter instead.

## How can Looker integrate with data orchestration tools like Apache Airflow?

Looker can be integrated with data orchestration tools like Apache Airflow through custom scripts or using Looker's API to trigger data processes or update Looker models.

The usual integration points: Airflow triggers a PDT rebuild by hitting the Looker API (`start_pdt_build`/`check_pdt_build`), or the ETL writes a datagroup trigger table that invalidates Looker cache, or Airflow runs `looker-cli`/`gcloud` to deploy LookML after a dbt run.

In [ ]:
# ILLUSTRATIVE - requires looker_sdk: install to run, or read on.
# Airflow task: rebuild a PDT after the warehouse load, then fail the DAG if it errors
import looker_sdk
import time

def rebuild_pdt(model: str, view: str) -> None:
    sdk = looker_sdk.init40()
    materialization = sdk.start_pdt_build(model_name=model, view_name=view)
    while True:
        state = sdk.check_pdt_build(materialization.materialization_id)
        if state.status in {"complete", "error", "killed"}:
            break
        time.sleep(10)
    if state.status != "complete":
        raise RuntimeError(f"PDT {view} build {state.status}: {state.message}")

The cleanest integration is indirect: the pipeline writes a small audit table, and a Looker datagroup watches it. No API call, no credential sharing, and cache invalidation follows the data.

```text
dbt run -> INSERT INTO analytics.etl_audit (table_name, etl_run_at) VALUES ('fct_orders', CURRENT_TIMESTAMP())
Looker datagroup: SELECT MAX(etl_run_at) FROM analytics.etl_audit WHERE table_name = 'fct_orders'
```

Gotcha: the trigger row must change only after the load commits, or Looker invalidates the cache before the new data lands and serves a mix of old and new.

Common mistake: having Airflow call the deploy API on every run. Deploy is a code promotion; cache invalidation is a data event. Keep them separate.

## What is the purpose of Looker's SQL Runner?

Looker's SQL Runner provides a SQL interface within Looker for users to write and execute custom SQL queries against the connected data sources.

SQL Runner is an admin/developer tool: run SQL against any connection, inspect schemas, create and drop tables, and run Explain. It is powerful and dangerous — a developer with SQL Runner can write to the warehouse — so restrict the `sql_runner` permission to a small group and prefer LookML for anything reusable.

```sql
-- SQL Runner is useful to validate a derivation before modeling it
SELECT country, COUNT(*) AS orders, ROUND(AVG(amount), 2) AS aov
FROM analytics.fct_orders
WHERE created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY country
ORDER BY orders DESC;
```

SQL Runner is a developer and admin tool: run SQL on any connection, browse schemas, create and drop tables, and run Explain against the generated plan.

```sql
-- validate a derivation before modeling it
SELECT country, COUNT(*) AS orders, ROUND(AVG(amount), 2) AS aov
FROM analytics.fct_orders
WHERE created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY country;
```

Gotcha: SQL Runner bypasses LookML entirely, so no access filter, no access grant, and no governed metric applies. It is the largest privilege in the application.

Common mistake: leaving it enabled for everyone. Restrict the `sql_runner` permission to developers who also hold a warehouse role you are willing to audit.

## What is the difference between Looker's explore and join types?

Looker's explore defines a specific view of data, while join types determine how multiple tables are joined together to create a consolidated view of the data.

Join `type` (`left_outer`, `inner`, `full_outer`, `cross`) and `relationship` (`one_to_one`, `many_to_one`, `one_to_many`, `many_to_many`) are separate declarations in LookML; `relationship` is what lets Looker apply symmetric aggregates and choose the correct join direction, while `type` is the SQL join. Joining a `one_to_many` fanout into a measure-bearing explore without symmetric aggregates inflates every `type: sum`.

```text
join: order_items {
  type: left_outer
  relationship: one_to_many      # fanout! metrics on orders must be fanout-safe
  sql_on: ${orders.order_id} = ${order_items.order_id} ;;
}
```

`relationship` tells Looker the cardinality; `type` tells the warehouse the SQL join. Looker uses `relationship` to decide join direction and whether to wrap a measure in a symmetric aggregate.

```text
relationship: many_to_one  -> safe to join fact -> dimension; no fanout
relationship: one_to_many  -> fanout; measures on the base view need symmetric aggregates
relationship: many_to_many -> needs a bridge table; otherwise rows multiply
```

Gotcha: a `left_outer` join to a dimension with duplicate keys fans out even though the relationship is declared many-to-one. The declaration does not fix the data; it only fixes deduplication when a primary key exists.

Common mistake: joining a lookup table with no unique key. Looker cannot protect you when the "dimension" is not unique.

## What is the purpose of Looker's content validation feature?

Looker's content validation feature allows users to validate and ensure the accuracy of data displayed in dashboards, reports, and visualizations.

Content Validator runs every saved Look and dashboard against the current model and reports broken fields, removed dimensions/measures, and errored queries — the go-to tool before deploying a LookML change or migrating a database. It does not validate the numbers are *correct*, only that the queries still run.

```bash
# Content validation via the API is the CI gate before a LookML deploy
curl -s -X POST -H "Authorization: token $LOOKER_TOKEN" \
  "https://acme.looker.com/api/4.0/content_validation" \
  | python -c "import sys,json; d=json.load(sys.stdin); print('errors:', len(d.get('errors',[])), 'checked:', d.get('total'))"
```

Content validation walks every saved Look and dashboard and reports fields that no longer exist, models that fail to compile, and queries that error. It is the release gate, not a data-quality check.

```text
POST /api/4.0/content_validation
-> { total: 412, errors: [ { content_id, message } ] }
```

Gotcha: it validates resolvability, not numbers. A measure whose SQL changed but whose name survived passes validation and still reports a different number.

Common mistake: running validation only after deploy. Run it against the development branch as a CI step, before merge.

## Can you explain Looker's dynamic data masking feature?

Looker does not provide native dynamic data masking capabilities. Data masking should be implemented at the database or data source level before connecting to Looker.

Looker can approximate masking with Liquid + user attributes, but it is model-level obfuscation, not a database security control, and the raw value still appears in SQL Runner or a downloaded CSV for anyone with access. Real masking belongs in the warehouse (policy tags, Snowflake masking policies) or in an access-granted secured view.

```text
dimension: email_masked {
  sql: CASE
         WHEN '{{ _user_attributes["pii_access"] }}' = 'yes' THEN ${TABLE}.email
         ELSE CONCAT(LEFT(${TABLE}.email, 2), '***@***')
       END ;;
  required_access_grants: [view_pii_masked]
}
```

Looker has no database-grade masking. The closest model-level controls are an access-granted field that returns a masked expression, and a Liquid CASE driven by a user attribute.

```text
dimension: email_masked {
  sql: CASE WHEN '{{ _user_attributes["pii_access"] }}' = 'yes'
            THEN ${TABLE}.email
            ELSE CONCAT(LEFT(${TABLE}.email, 2), '***@***') END ;;
  required_access_grants: [view_pii_masked]
}
```

Gotcha: the raw column is still in the table. Anyone with SQL Runner or a direct warehouse credential sees it. Real masking belongs in the warehouse (policy tags, masking policies) or in a secured view.

Common mistake: shipping a masked dimension while the unmasked one remains in the same explore, which makes the mask cosmetic.

## What is the purpose of Looker's Liquid syntax in LookML?

Looker's Liquid syntax is used within LookML to dynamically generate SQL or control the behavior of dimensions, measures, or filters based on conditions or user inputs.

Liquid runs at query-generation time and can read parameters, user attributes, filters, and dates. It is the mechanism behind parameterized dimensions, date-range-aware PDT SQL, and conditional field masking.

```text
dimension: dynamic_metric {
  sql:
    {% if _parameter_value == 'revenue' %} ${total_revenue}
    {% elsif _parameter_value == 'orders' %} ${order_count}
    {% else %} NULL {% endif %} ;;
}
```


```text
# Liquid can read the filter the user selected, e.g. for a date-bounded PDT
derived_table: {
  sql:
    SELECT * FROM analytics.events
    WHERE event_date BETWEEN {% date_start event_date %} AND {% date_end event_date %} ;;
}
```

Liquid runs at query-generation time and can read four kinds of state: user attributes, parameter values, the selected filters, and identifiers such as the model or explore name.

```text
{{ _user_attributes["region"] }}        -> row/field security, embed attributes
{{ _parameter_value }}                   -> the value of a parameter
{% date_start orders.created_date %}     -> the start of the user's date filter
{{ _model._name }} / {{ _explore._name }}-> context for shared logic
```

Worked example: a date-bounded derived table that only materializes the range the user asked for.

```text
derived_table: {
  sql:
    SELECT * FROM analytics.events
    WHERE event_date BETWEEN {% date_start event_date %} AND {% date_end event_date %} ;;
}
```

Gotcha: Liquid output is interpolated into SQL text. Looker escapes user-attribute values used in filters, but a value interpolated directly into a `sql:` expression is raw. Never paste unvalidated user input into SQL.

Common mistake: changing the generated SQL per user in a way that defeats cache sharing for everyone. Use Liquid where the value genuinely differs by security context.

## What is the purpose of Looker's Development Mode?

Looker's Development Mode allows developers to make changes to LookML models without affecting the production environment. It provides a safe testing environment before deploying changes.

Development Mode creates a branch, keeps the IDE on dev until changes are committed, and can show a banner on content ("You are viewing in development mode"). Production is unaffected until Deploy. The important discipline: never validate against production data only — use a staging branch/environment and content validation before merging.

Development Mode separates editing from serving: it holds a personal branch, keeps the IDE pointed at those edits, and shows a banner so nobody mistakes dev content for production.

```text
development mode ON  -> edit LookML on your branch; explores use your branch
development mode OFF -> edits go through commit; content uses the deployed model
Deploy               -> promote the committed model to Production
```

Gotcha: dashboard content is not versioned with the branch. A tile built in dev mode against a dev-only field is broken the moment the field is not merged.

Common mistake: validating only against production data and skipping content validation on the branch, then discovering broken tiles after deploy.

## How do you design and rebuild PDTs for performance?

A PDT earns its cost only when it turns a repeated expensive computation into a cheap read. The design decision is which persistence trigger to use.

```text
persist_for: "24 hours"        time-based; rebuilds 24h after the last build
datagroup_trigger: orders_default  data-based; shares one ETL signal across PDTs
sql_trigger: SELECT MAX(loaded_at) FROM ...   data-based, local to one derived table
incremental: yes               append new rows/partitions instead of a full rebuild
```

Prefer `datagroup_trigger` when the ETL can signal completion, `persist_for` only when it cannot, and incremental when the source is append-mostly and large.

Design rules that hold up in production.

```text
1. materialize the expensive part only: a big join or aggregation reused by many tiles
2. cluster_keys / sort on the columns Looker filters and groups by
3. one datagroup per freshness tier; never one trigger for every PDT
4. use the designer/reader schema split so a rebuild never breaks readers
5. watch build history; a PDT rebuilding every few minutes is mis-triggered
```

Worked example: `fct_events` is 4B rows and every dashboard asks for events by day and name. A daily PDT reduces each query from a 4B-row scan to a 90-day read of a small aggregate, and clustering on `event_date` keeps even that cheap.

Common mistake: a short `persist_for` on a large base table. If the build takes 20 minutes and `persist_for` is 1 hour, the table is rebuilding for a third of its life and readers queue behind it. Measure the build time and set the interval well above it, or switch to a data trigger.

## How does aggregate awareness and symmetric-aggregate cost work?

Aggregate awareness lets Looker substitute a pre-aggregated rollup when the requested fields, filters and measures are all present in it. The user sees no difference; the warehouse reads far less.

```text
explore: orders {
  aggregate_table: daily_rollup {
    query: {
      dimensions: [orders.created_date, orders.country]
      measures: [orders.total_revenue, orders.order_count]
    }
    materialization: { datagroup_trigger: orders_default }
  }
}
```

The rollup is itself a persistent table. Looker maintains it from the same model, so there is no second definition to drift.

Matching is strict. A query uses the rollup only when every requested dimension and measure exists in it and every filter can be expressed from its columns. Adding one dimension the rollup lacks, such as `orders.status`, sends the query back to the full fact table.

```text
requested: date + country + total_revenue           -> rollup used
requested: date + country + status + total_revenue  -> full table
```

Symmetric aggregates are fanout-safe but not free: Looker wraps the measure in a distinct-key subquery with its own `GROUP BY`, so the warehouse does more work than a naive sum. Use them where fanout exists rather than everywhere.

Common mistake: adding `sql_distinct_key` to a measure on a view that is already at the right grain. The extra subquery buys nothing and slows every query that includes the measure.

## How do you choose a caching strategy for looks and dashboards?

Three controls, in order of preference.

```text
datagroup + persist_with   invalidates when the ETL signal changes (best)
max_cache_age              ceiling when no trigger fires; a fallback, not a strategy
clear cache and refresh    manual bypass for a just-loaded row
```

A datagroup ties freshness to the data; `max_cache_age` alone means a dashboard can serve pre-load results until the timer expires.

The cache key includes the security context, so row-level security multiplies cache entries. Ten regions and one dashboard means up to ten cached results, not one.

```text
cache entries = distinct generated SQL x distinct security contexts
cost          = entries x rebuild frequency x warehouse query cost
```

Worked example: a finance dashboard runs 400 times a day. With a datagroup that changes once after the evening load, the warehouse sees a couple of queries instead of 400; the other 398 are cache hits.

Common mistake: a long `max_cache_age` with no datagroup. After a late ETL fix, users stare at the old number until the timer runs out, and the first person to clear cache pays the full cost alone.

## How do you diagnose a slow Looker query?

Diagnose in this order, and stop as soon as the cause is clear.

```text
1. reproduce in the Explore and open the SQL tab
2. read the generated SQL against the intent: SELECT *, missing partition
   predicate, a function wrapping the partition column, an extra join
3. get the query id and read warehouse history: bytes scanned, partitions,
   spill, queue time
4. check fanout: compare result row count with the base grain count
5. check dimensions: each extra dimension changes GROUP BY and defeats rollups
6. only then change the model (PDT, aggregate_table, access_filter, always_filter)
```

A quick fanout test: compare `COUNT(*)` with `COUNT(DISTINCT primary_key)` in the generated query. A ratio above 1 means a join multiplied the base rows, and any `type: sum` on the base view is inflated unless it has a distinct key.

Common mistake: tuning the warehouse first. If the model generated a cartesian join or a full scan, a bigger warehouse makes the same mistake faster and more expensive.

## Things to remember: key numbers and limits


| Topic | Fact | Why it matters |
|---|---|---|
| Compute | Looker stores no data; every tile generates SQL against the warehouse | the warehouse's bytes-scanned bill is the real cost driver |
| Modeling unit | model -> explore -> view -> join; dimensions (grouping) and measures (aggregates) | fanout and symmetric aggregates live at the join definition |
| Fanout | a one-to-many join inflates sums unless symmetric aggregates kick in | wrong aggregates are silent and look correct on small samples |
| PDT | persistent derived table built in the warehouse on a datagroup/persistence trigger | great for expensive rollups; not a replacement for warehouse marts |
| Caching | queries cache per datagroup TTL; dashboard tiles share cached results | stale-looking dashboards are usually a datagroup mismatch |
| Security | model sets -> roles; access grants and access filters for column/row control | row security enforced in generated SQL, so it cannot leak in an explore |
| Development | Development Mode + Git branches + content validator | LookML changes are code: review, validate, deploy |
| Scheduling | deliveries and alerts evaluate on cache/PDT rebuilds | schedule after the warehouse ETL finishes, not against a wall clock |


## Things to remember: semantics to memorize


* **Semantic layer first.** Every metric lives once as a LookML measure; if the same number is computed in dashboard tiles, it is a bug factory.
* **Aggregate awareness / symmetric aggregates handle fanout.** Declare join relationships (`one_to_one`, `many_to_one`, ...) accurately and Looker rewrites the SQL to avoid double counting.
* **PDTs persist; ephemerals recompute.** Use a PDT for anything queried by many tiles; use a datagroup keyed to the source table's freshness.
* **The cache is shared and content-keyed.** Two tiles with the same query hit the same cache — dashboards get cheaper as they converge on shared explores.
* **Looker is not the ETL tool.** Heavy transformation belongs upstream (dbt, warehouse marts); Looker models the result, it does not manufacture it.


## Things to remember: mistakes that keep coming back


* Joining on a non-unique key without checking the relationship — fanout inflates revenue with no error message.
* Building a PDT to paper over a missing warehouse mart — the PDT rebuild storm just moves the cost.
* Leaving the default 1-hour datagroup on a table that loads nightly — stale dashboards all morning.
* 50 unfiltered tiles on one dashboard — each is a warehouse query; use aggregate awareness and fewer, shared explores.
* Row-level security done in application code on top of Looker exports — use access filters so the SQL itself is scoped.
* Editing LookML directly on production without the content validator — broken explores discovered by users.
* Using SQL Runner queries as reports — unversioned, unmodeled, and uncached; promote to a view.


## Real world use cases for Looker

### 1. Marketing attribution for a multi-brand retailer

Business context: three brands share one data team; marketing wants CAC, ROAS, and channel mix by brand and campaign, refreshed daily, with each brand's team seeing only its own data. Volume: 12 ad platforms, 90M impressions/day, 14M sessions/day, 220K orders/day.

```text
ad APIs + clickstream + CRM -> dbt marts (finance.fct_marketing_daily)
   -> LookML: view marketing_daily + explore marketing
   -> access_filter by brand_id (user attribute from SSO groups)
   -> dashboard: CAC/ROAS tiles + campaign drill + alert on spend > plan
```

```text
explore: marketing_daily {
  access_filter: {
    field: marketing_daily.brand_id
    user_attribute: brand_id
  }
}
```


Why this design: attribution logic lives once in LookML, so the three brand dashboards cannot drift; the brand row-level filter is a single user attribute fed from SSO groups, and the ratio measures are computed from summed components (never averaged), which is the difference between a defensible CAC and a misleading one.

### 2. Embedded analytics for a B2B SaaS platform

Business context: a vertical SaaS vendor sells usage analytics to 1,100 tenant companies inside its product; tenants only see their own data, and the vendor wants zero per-tenant modeling work. Volume: 2.5B events/month, 400 concurrent dashboard users, 60 tenants on premium isolated warehouses.

```text
product events -> warehouse (shared fct_usage + per-tenant schemas for premium)
   -> LookML model per tier
   -> SSO embed URLs from the product backend (signed with embed secret)
   -> tenant-scoped user attributes: tenant_id, plan, can_see_cost
```

In [ ]:
import base64
# Backend issues one signed embed URL per request; attributes carry the tenant
import hashlib
import hmac
import time
import urllib.parse

def signed_embed(secret: str, path: str, user: dict) -> str:
    params = {"embed_domain": "https://app.acme.com", **user}
    url = f"https://acme.looker.com{path}?{urllib.parse.urlencode(params)}"
    url += f"&nonce={time.time()}"
    url += "&signature=" + urllib.parse.quote(
        base64.b64encode(
            hmac.new(secret.encode(), url.encode(), hashlib.sha1).digest()
        ).decode()
    )
    return url

print(signed_embed("s3cr3t", "/embed/dashboards/usage",
                   {"tenant_id": "northwind", "plan": "premium"}))

Why this design: SSO embed plus a `tenant_id` user attribute means the model enforces isolation regardless of what the host application sends, one dashboard serves every tenant, and premium tenants can be routed to dedicated warehouses without changing the LookML or the product code beyond the attribute.

### 3. Regulatory reporting for a fintech

Business context: a lender must produce monthly capital and exposure reports with an auditable definition, regional row-level access, and a delivery to regulators and internal risk. Volume: 180M loan-month rows, 40 years of history, 12 report variants.

```text
core banking CDC -> warehouse (risk.fct_exposure, risk.dim_loan)
   -> LookML views with required_access_grants on PII and certified metrics
   -> dashboards + scheduled PDF/CSV deliveries to risk and regulators
   -> content validation in CI; every deliverable versioned
```

```text
measure: exposure_at_default {
  label: "Exposure at Default (EAD)"
  description: "Certified for regulatory reporting. Source: risk.fct_exposure.ead. Owner: risk-modeling."
  tags: ["certified", "regulatory"]
  type: sum
  sql: ${TABLE}.ead ;;
}
```


Why this design: regulators require a single documented definition and an audit trail, so the metric, its description, and its tags live in Git where every change is reviewed; access grants keep borrower PII out of the reporting model except for the risk group, and scheduled Advanced deliveries with `run_as_recipient` ensure each region's report is filtered by that region's permissions.

### 4. Product analytics and retention for a consumer marketplace

Business context: the growth team wants weekly cohorts, retention curves, and feature adoption by acquisition channel, without asking the data team for every cut. Volume: 900M product events/month, 30M users, 24-month retention horizon.

```text
events -> warehouse (mart_user_activity_daily: user_id, ds, feature, channel)
   -> LookML: explore with cohort/retention derived table + parameters
   -> dashboard: cohort heatmap, retention curve, feature adoption
```

```text
view: cohort_activity {
  derived_table: {
    sql:
      WITH first_seen AS (
        SELECT user_id, MIN(event_date) AS cohort_date
        FROM marts.user_activity_daily GROUP BY 1
      )
      SELECT a.user_id, f.cohort_date,
             DATE_DIFF(a.event_date, f.cohort_date, DAY) AS day_number,
             a.feature, a.channel
      FROM marts.user_activity_daily a
      JOIN first_seen f USING (user_id) ;;
    datagroup_trigger: activity_default
    cluster_keys: ["cohort_date", "feature"]
  }
```


Why this design: the cohort definition (first event date) is computed once in a persisted derived table, so every analyst gets the same retention number; clustering on cohort and feature keeps the weekly rebuild and the interactive heatmap affordable, and the parameterized explore lets growth self-serve cuts without new tiles.

### 5. Retail operations command center

Business context: a grocery chain monitors store operations in near real time during trading hours; a store manager sees only their store, a regional director only their region, and the exec team sees the chain. Volume: 1,200 stores, 40M transactions/day, 5-minute micro-batches.

```text
POS -> streaming/micro-batch -> warehouse (ops.store_trading_5m)
   -> LookML explores: store_ops (short cache) + chain_ops (aggregate)
   -> dashboard: sales vs plan, basket size, queue alerts
   -> access_filter: store_id for managers, region for directors
```

```text
explore: store_ops {
  persist_with: ops_5m          # max_cache_age: 5 minutes
  always_filter: { filters: [store_ops.business_date: "today"] }
  access_filter: [
    { field: store_ops.store_id, user_attribute: store_id },
    { field: store_ops.region,   user_attribute: region }
  ]
}
```


Why this design: a 5-minute cache with a today-only filter keeps the live dashboard cheap enough to serve 1,200 managers, while the region/chain views roll up from the same governed measures so a store manager and the COO cannot see different definitions of net sales; stacking two access filters lets one explore serve three levels of the org without three dashboards.

The five cases below are chosen to exercise the whole model, not just the visualisation: row-level security by user attribute, SSO embedding, certified regulated metrics, a persisted derived table with clustering, and a short-cache operational explore.

Read each one for the design decision it illustrates. The LookML is abridged but uses real parameters, and the "why this design" paragraph names the tradeoff that was accepted. The recurring theme is that the model, not the dashboard, carries governance, freshness and cost.

## Looker exercises

These exercises use only the Python standard library and model the decisions behind a Looker model: fanout, PDT choice, caching, aggregate awareness, row-level security, incremental builds and cost. Each one names the task, gives a hint, states the expected result, then prints and asserts the answer.

The arithmetic mirrors what you would read from the SQL tab of an Explore, the PDT build history, and the warehouse query history. Treat the asserted numbers as a check on the formula, not a prediction of a real instance. Run them with `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

### Try it 1. Detect fanout and fix it with a symmetric aggregate

**Task.** Two orders are joined to a one-to-many item table. Compute the naive sum across the join, the symmetric-aggregate sum, and the inflation ratio.

**Hint.** The naive sum multiplies each order amount by its item count; the symmetric aggregate keeps one value per distinct order id.

**Expected result.** naive = 3000.0, symmetric = 1000.0, inflation = 3.0x

In [ ]:
orders = [
    {"order_id": "o1", "amount": 400.0},
    {"order_id": "o2", "amount": 600.0},
]
items_per_order = {"o1": 3, "o2": 3}

# A naive SUM(amount) over the join repeats each order once per item row.
naive = sum(o["amount"] * items_per_order[o["order_id"]] for o in orders)
# Symmetric aggregates sum one value per distinct primary key.
symmetric = sum(o["amount"] for o in orders)
inflation = naive / symmetric

print(f"naive={naive:.1f} symmetric={symmetric:.1f} inflation={inflation:.1f}x")
assert symmetric == 1000.0
assert round(inflation, 1) == 3.0


### Try it 2. Choose a PDT type from freshness and reuse

**Task.** Classify each candidate as ephemeral, incremental, datagroup_trigger or persist_for from its refresh need, reuse rate and daily new-row share.

**Hint.** Not reused: inline it. Large and append-heavy: incremental. Needs data-driven freshness: datagroup_trigger. Otherwise a wall-clock persist_for.

**Expected result.** orders_by_day -> incremental, daily_summary -> datagroup_trigger, adhoc_slice -> ephemeral

In [ ]:
def choose_pdt(reuse_per_hour, new_row_share, needs_data_trigger, append_only):
    if reuse_per_hour < 1:
        return "ephemeral"
    if append_only and new_row_share < 0.05:
        return "incremental"
    if needs_data_trigger:
        return "datagroup_trigger"
    return "persist_for"


cases = {
    "orders_by_day": choose_pdt(40, 0.01, True, True),
    "daily_summary": choose_pdt(12, 0.30, True, False),
    "adhoc_slice": choose_pdt(0, 1.00, False, False),
}
print(cases)
assert cases["orders_by_day"] == "incremental"
assert cases["daily_summary"] == "datagroup_trigger"
assert cases["adhoc_slice"] == "ephemeral"


### Try it 3. Compute the cache hit benefit from query frequency

**Task.** A dashboard is requested 400 times a day with a once-daily datagroup invalidation. Compute the cache hit rate, the warehouse queries served, and the cost saved at 0.02 dollars per query.

**Hint.** With one datagroup change per day, the first request of each cache window misses and the rest hit. Saved cost = hits x cost per query.

**Expected result.** hits = 399, hit rate = 99.75%, saved = 7.98 dollars

In [ ]:
requests_per_day = 400
cache_windows_per_day = 1
cost_per_query = 0.02

# One miss per cache window: the first request after each invalidation.
misses = cache_windows_per_day
hits = requests_per_day - misses
hit_rate = hits / requests_per_day
saved = hits * cost_per_query

print(f"hits={hits} hit_rate={hit_rate:.4f} saved={saved:.2f}")
assert hits == 399
assert round(hit_rate, 4) == 0.9975
assert round(saved, 2) == 7.98


### Try it 4. Pick an aggregate-awareness table from query patterns

**Task.** Given three candidate rollups and the field sets analysts request, choose the rollup that covers the most requests.

**Hint.** A request is covered only when every dimension and measure it asks for is present in the rollup.

**Expected result.** daily_rollup covers 3 of 4 requests

In [ ]:
queries = [
    {"dims": {"created_date", "country"}, "measures": {"total_revenue"}},
    {"dims": {"created_date"}, "measures": {"total_revenue", "order_count"}},
    {"dims": {"created_date", "country"}, "measures": {"total_revenue", "order_count"}},
    {"dims": {"created_date", "status"}, "measures": {"total_revenue"}},
]
rollups = {
    "daily_rollup": {"dims": {"created_date", "country"},
                     "measures": {"total_revenue", "order_count"}},
    "country_rollup": {"dims": {"country"}, "measures": {"total_revenue"}},
}


def covers(rollup, query):
    return (query["dims"] <= rollup["dims"]
            and query["measures"] <= rollup["measures"])


coverage = {name: sum(covers(r, q) for q in queries) for name, r in rollups.items()}
best = max(coverage, key=coverage.get)
print(coverage, "best:", best)
assert coverage["daily_rollup"] == 3
assert best == "daily_rollup"


### Try it 5. Compute row-level filter correctness from sample users

**Task.** Given a sample orders table and per-user region attributes, compute the visible rows per user and flag any user whose result leaks another region.

**Hint.** Replicate the access_filter: a user sees only rows whose region equals their attribute. A missing attribute means no rows.

**Expected result.** alice sees 2, bob sees 1, carol sees 0, leaks = []

In [ ]:
orders = [
    {"order_id": "o1", "region": "north", "amount": 10.0},
    {"order_id": "o2", "region": "south", "amount": 20.0},
    {"order_id": "o3", "region": "north", "amount": 30.0},
]
users = {"alice": "north", "bob": "south", "carol": None}


def visible(user_region):
    if user_region is None:
        return []
    return [o for o in orders if o["region"] == user_region]


results = {u: visible(r) for u, r in users.items()}
leaks = [u for u, rows in results.items()
         if any(o["region"] != users[u] for o in rows)]
counts = {u: len(rows) for u, rows in results.items()}
print(counts, "leaks:", leaks)
assert counts == {"alice": 2, "bob": 1, "carol": 0}
assert leaks == []


### Try it 6. Size an incremental PDT from new rows per day

**Task.** A full events table holds 900M rows and grows by 5M rows a day. Compute the incremental append size, the full-rebuild size, and the reduction for a 30-day retention rebuild.

**Hint.** The incremental build appends new rows since the last build; the full rebuild reprocesses the whole retained table.

**Expected result.** incremental = 5,000,000 rows; full = 1,050,000,000 rows; reduction = 210x

In [ ]:
total_rows = 900_000_000
new_rows_per_day = 5_000_000
retention_days = 30

retained_rows = total_rows + new_rows_per_day * retention_days
incremental_rows = new_rows_per_day
reduction = retained_rows / incremental_rows

print(f"incremental={incremental_rows:,} full={retained_rows:,} reduction={reduction:.0f}x")
assert incremental_rows == 5_000_000
assert retained_rows == 1_050_000_000
assert round(reduction) == 210


### Try it 7. Estimate warehouse cost saved by caching

**Task.** A dashboard runs 5,000 queries a day at 0.15 dollars each. With a 92 percent cache hit rate, compute the queries billed, the daily cost with and without cache, and the monthly saving.

**Hint.** Only cache misses are billed. The saving is the difference between the uncached and cached daily cost, times 30 days.

**Expected result.** billed = 400, cached cost = 60.00, uncached cost = 750.00, monthly saving = 20700.00

In [ ]:
queries_per_day = 5_000
cost_per_query = 0.15
hit_rate = 0.92

billed = queries_per_day * (1 - hit_rate)
cached_cost = billed * cost_per_query
uncached_cost = queries_per_day * cost_per_query
monthly_saving = (uncached_cost - cached_cost) * 30

print(f"billed={billed:.0f} cached={cached_cost:.2f} "
      f"uncached={uncached_cost:.2f} monthly_saving={monthly_saving:.2f}")
assert round(billed) == 400
assert round(cached_cost, 2) == 60.00
assert round(monthly_saving, 2) == 20700.00


### Try it 8. Choose persist_for versus a datagroup trigger

**Task.** The ETL completes four times a day and the PDT takes 20 minutes to build. Compute the rebuilds per day for a 6-hour persist_for versus a datagroup trigger, and the share of the day spent building.

**Hint.** persist_for rebuilds whenever its interval elapses, independent of the load. A datagroup rebuilds once per trigger change, capped by max_cache_age.

**Expected result.** persist_for = 4 builds, datagroup = 4 builds, build time = 80 minutes (5.6% of the day)

In [ ]:
etl_loads_per_day = 4
persist_minutes = 360
build_minutes = 20

persist_builds = (24 * 60) // persist_minutes
datagroup_builds = etl_loads_per_day
build_time = datagroup_builds * build_minutes
build_share = build_time / (24 * 60)

print(f"persist_for={persist_builds} datagroup={datagroup_builds} "
      f"build_time={build_time}min share={build_share:.3f}")
assert persist_builds == 4
assert datagroup_builds == 4
assert build_time == 80
assert round(build_share, 3) == 0.056
